In [ ]:
import math
import pandas as pd
import os
import pyarrow.parquet as pq
import pyarrow as pa
from glob import  glob
from collections import Counter
from pyarrow.parquet import ParquetFile

# Speicherort der Datei
OUTPUT_FILE = "../data/processed/"

files = sorted(
    glob(os.path.join(OUTPUT_FILE,"nyc_taxi_cleaned1.parquet"))
)
#files = sorted( glob(os.path.join(OUTPUT_FILE,"*.parquet")))

# Statisitk Werte :
#Initialization
total_fare = 0
trip_count = 0
global_min = float("inf")
global_max = float("-inf")
counter = Counter()

n=0
mean = 0
M2 = 0
# Korrelation
mean_x = 0
mean_y = 0
M2_x = 0
M2_y = 0
C_xy = 0

BIN_SIZE = 5
histogram = Counter()

for file in files :
    print(f"\nTraitement du fichier : {file}")

    parquet_file = ParquetFile(file)
    for batch in parquet_file.iter_batches(batch_size=100_000):
        df = batch.to_pandas()

        total_fare += df["fare_amount"].sum()
        trip_count += len(df)
        global_min = min(global_min, min(df["fare_amount"]))
        global_max = max(global_max, max(df["fare_amount"]))
        counter.update(df["hour"])

        #Varianz, Sdt (Welford-Algorithmus)
        values = df["fare_amount"].dropna()
        for x in  values:

            n+=1
            delta = x-mean
            mean += delta/n
            delta2 = x-mean
            M2 = delta * delta2

            bucket = int(x // BIN_SIZE)
            histogram[bucket]+=1

        #Korrelation
        subset = df[
            ["trip_distance", "fare_amount"]
            ].dropna()

        for x, y in zip(
            subset["trip_distance"],
            subset["fare_amount"]
        ):

            n += 1

            dx = x - mean_x
            mean_x += dx / n

            dy = y - mean_y
            mean_y += dy / n

            M2_x += dx * (x - mean_x)
            M2_y += dy * (y - mean_y)

            C_xy += dx * (y - mean_y)

var_x = M2_x / (n - 1)
var_y = M2_y / (n - 1)

cov_xy = C_xy / (n - 1)

corr = cov_xy / math.sqrt(var_x * var_y)

mean1 = total_fare/trip_count

variance = M2/(n-1)
std_dev = math.sqrt(variance)

print(f"\nSumme = {total_fare}")
print(f"\nAnzahl = {trip_count}")
print(f"Mean = {round(mean1,2)}")
print(f"Min = {global_min}")
print(f"Max = {global_max}")
print(counter.most_common(5))

print(f"Moyenne : {mean:.2f}")
print(f"Variance : {variance:.4f}")
print(f"Std Dev : {std_dev:.2f}")

print(f"Korrelation : {corr:.4f}")


In [26]:
#Histogram
for bucket in sorted(histogram):

    start = bucket * BIN_SIZE
    end = start + BIN_SIZE

    '''print(
        f"{start:3d}-{end:3d} : "
        f"{histogram[bucket]}"
    )'''

#Exportieren Histogramm
hist_df = pd.DataFrame([
    {
        "bin_start": b * BIN_SIZE,
        "bin_end": (b + 1) * BIN_SIZE,
        "count": c
    }
    for b, c in histogram.items()
])

hist_df.to_csv(
    "fare_histogram.csv",
    index=False
)

-1810--1805 : 1
-1000--995 : 1
-980--975 : 2
-960--955 : 1
-900--895 : 4
-860--855 : 1
-850--845 : 1
-835--830 : 1
-830--825 : 2
-805--800 : 1
-800--795 : 1
-700--695 : 5
-680--675 : 1
-655--650 : 1
-650--645 : 1
-635--630 : 1
-620--615 : 1
-600--595 : 3
-580--575 : 1
-565--560 : 1
-550--545 : 1
-545--540 : 1
-510--505 : 1
-505--500 : 1
-500--495 : 7
-495--490 : 1
-470--465 : 1
-460--455 : 1
-450--445 : 1
-430--425 : 4
-425--420 : 2
-420--415 : 4
-415--410 : 1
-400--395 : 7
-390--385 : 1
-385--380 : 1
-380--375 : 1
-375--370 : 2
-370--365 : 1
-360--355 : 2
-355--350 : 1
-350--345 : 4
-345--340 : 2
-340--335 : 4
-335--330 : 2
-330--325 : 1
-325--320 : 4
-320--315 : 1
-315--310 : 2
-310--305 : 4
-305--300 : 11
-300--295 : 17
-295--290 : 2
-290--285 : 6
-285--280 : 1
-280--275 : 6
-275--270 : 1
-270--265 : 6
-265--260 : 6
-260--255 : 7
-255--250 : 5
-250--245 : 17
-245--240 : 8
-240--235 : 5
-235--230 : 8
-230--225 : 8
-225--220 : 6
-220--215 : 9
-215--210 : 5
-210--205 : 10
-205--200 : 6

In [33]:
total_count = sum(histogram.values())
median_pos = total_count / 2
running = 0

for bucket in sorted(histogram):

    running += histogram[bucket]

    if running >= median_pos:

        median_bucket = bucket
        break

median_estimate = (
    median_bucket * BIN_SIZE
    + BIN_SIZE / 2
)
print(median_bucket)
print(median_estimate)

2
12.5


In [30]:
#Quantile
def histogram_quantile(
    histogram,
    total_count,
    quantile,
    bin_size
):

    target = total_count * quantile

    running = 0

    for bucket in sorted(histogram):

        running += histogram[bucket]

        if running >= target:

            return (
                bucket * bin_size
                + bin_size / 2
            )


In [32]:
q25 = histogram_quantile(
    histogram,
    total_count,
    0.25,
    BIN_SIZE
)

q50 = histogram_quantile(
    histogram,
    total_count,
    0.50,
    BIN_SIZE
)

q75 = histogram_quantile(
    histogram,
    total_count,
    0.75,
    BIN_SIZE
)

print(f" {q25} {q50} {q75}")

 7.5 12.5 22.5
